# Pneumonia Screening from Chest X-rays (AI + Health, project 4.1.1)

> **This project is a prototype and is not medical advice.**

**Who is it for?** A nurse or general doctor at a small clinic **with no radiologist on site**. Many chest
X-rays come in every day and a specialist reads them hours or days later. Our tool sorts the queue so the
scans most likely to show pneumonia get read first.

**What goes in, and what comes out?** A chest X-ray image goes in. Out comes a pneumonia probability and
one of three triage labels:

| Label | Meaning | What staff do |
|---|---|---|
| 🔴 **URGENT** | Model is confident it's pneumonia | Put it at the top of the reading queue |
| 🟡 **UNSURE** | Model isn't confident either way | A person must review it, as with urgent cases |
| 🟢 **LIKELY NORMAL** | Model is confident it's normal | Normal queue (it's still read by a clinician) |

**What we added (make it your own):**
1. **A real setting:** triage for a clinic with no radiologist, rather than plain classification.
2. **Handling unsure scans:** instead of forcing a yes/no answer, the tool marks scans in an "unsure" band
   for a person to review. We choose the band's limits on the validation set and measure on the test set
   what it does to missed cases.
3. **A threshold chosen for safety:** a missed pneumonia costs more than a false alarm, so we pick the
   threshold that catches at least 95% of pneumonia cases on validation data. We don't use the default 0.5.
4. **Visual interpretability:** Grad-CAM heatmaps show where in the X-ray the model "looks", and we check
   whether it focuses on the lungs or on image borders and markers (a sign of shortcut learning).
5. **Edge-case and noise robustness:** we measure how missed cases and false alarms change under noise, blur,
   rotation, low resolution and JPEG compression, and add an input check that refuses blank or non-X-ray images.
6. **Multi-modal metadata:** the nurse can enter age, oxygen level, breathing rate and danger signs. Simple
   rules based on WHO guidance can only *raise* the urgency of a scan, never lower it.
7. **Triage queue simulation:** we simulate a clinic's reading queue and measure how much sooner pneumonia
   cases get read when the model orders the queue, compared with first-come-first-served.
8. **Inference latency and edge deployment:** milliseconds per X-ray on GPU, CPU and ONNX Runtime, plus model size,
   to judge whether the tool could run on a cheap clinic computer.

**Notebook outline** (approximate run time on a free Colab T4 GPU in brackets)
1. Setup and data (about 1 min)
2. Baseline: "always predict pneumonia" (seconds)
3. Model 1: a small CNN trained from scratch (1–2 min)
4. Model 2: a fine-tuned pretrained ResNet-18 (about 2 min)
5. Choosing the threshold and the "unsure" band (seconds)
6. Final test results, with missed cases and false alarms reported separately (seconds)
7. Where it fails (seconds)
8. Visual interpretability with Grad-CAM (under 1 min)
9. Edge-case and noise robustness tests (about 1 min)
10. Multi-modal metadata: vital-sign rules (seconds)
11. Triage queue simulation (under 1 min)
12. Inference latency and edge deployment benchmarks (1–2 min)
13. Demo app with Gemini explanations (starts in under 1 min)

Every section prints how long it took (⏱), and the last cell prints the total run time.

## 1. Setup and data

In Colab, go to **Runtime → Change runtime type → T4 GPU** first. Training also works on CPU, just more slowly.

In [ ]:
!pip install -q medmnist gradio google-genai onnx onnxruntime onnxscript

In [ ]:
import os, random, re, time
NOTEBOOK_START = time.time()

class Timer:
    # Prints how long a section took:  with Timer("Section 8"): ...
    def __init__(self, name): self.name = name
    def __enter__(self): self.t0 = time.time(); return self
    def __exit__(self, *exc): print(f"\n⏱ {self.name} took {time.time() - self.t0:.1f} s")
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision
from sklearn.metrics import confusion_matrix, roc_auc_score
import medmnist
from medmnist import PneumoniaMNIST

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device, "| medmnist", medmnist.__version__)

# Which data to train on:
#   "medmnist": PneumoniaMNIST, 64x64 pixels, downloads by itself (default, fastest)
#   "kaggle":   the full-size Kaggle "Chest X-Ray Images (Pneumonia)" dataset by Paul Mooney (Kermany et al.),
#               resized to KAGGLE_IMG_SIZE. Same hospital and same children as PneumoniaMNIST, but sharper images.
DATA_SOURCE = os.environ.get("DATA_SOURCE", "medmnist")
KAGGLE_IMG_SIZE = 128
IMG_SIZE = 64 if DATA_SOURCE == "medmnist" else KAGGLE_IMG_SIZE
QUICK_RUN = os.environ.get("QUICK_RUN") == "1"   # only for fast testing; leave off for real results
EPOCHS_CNN = 1 if QUICK_RUN else 15
EPOCHS_RESNET = 1 if QUICK_RUN else 8

In [ ]:
from PIL import Image

def find_kaggle_folder():
    # Looks for the folder that contains train/NORMAL and test/NORMAL
    roots = ["/kaggle/input/chest-xray-pneumonia", "/kaggle/input/chest-xray-pneumonia/chest_xray",
             "/content/chest-xray-pneumonia", "/content/chest_xray", "chest_xray", "chest-xray-pneumonia"]
    for r in roots:
        for cand in (r, os.path.join(r, "chest_xray")):
            if os.path.isdir(os.path.join(cand, "train", "NORMAL")):
                return cand
    try:
        import kagglehub      # works in Colab/Kaggle when you are signed in to Kaggle
        path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
        for cand in (path, os.path.join(path, "chest_xray")):
            if os.path.isdir(os.path.join(cand, "train", "NORMAL")):
                return cand
    except Exception as e:
        print("Automatic Kaggle download did not work:", type(e).__name__, str(e)[:150])
    raise FileNotFoundError(
        "Kaggle dataset not found. In Kaggle: Add Data -> search 'Chest X-Ray Images (Pneumonia)' (Paul Mooney). "
        "In Colab: upload kaggle.json, then run  !pip install kagglehub  and re-run, or unzip the dataset to /content/chest_xray.")

def patient_id(filename):
    # Several X-rays can come from the same patient. Keeping a patient in ONE split avoids leakage.
    m = re.search(r"(person\d+)", filename) or re.search(r"(IM-\d+)", filename)
    return m.group(1) if m else filename

def load_folder(root, split, size):
    xs, ys, ids = [], [], []
    for cls, lab in [("NORMAL", 0), ("PNEUMONIA", 1)]:
        folder = os.path.join(root, split, cls)
        for f in sorted(os.listdir(folder)):
            if not f.lower().endswith((".jpeg", ".jpg", ".png")):
                continue
            im = Image.open(os.path.join(folder, f))
            im.draft("L", (size * 2, size * 2))                  # fast JPEG shrink while reading
            xs.append(np.asarray(im.convert("L").resize((size, size), Image.BILINEAR), dtype=np.uint8))
            ys.append(lab); ids.append(patient_id(f))
    return np.stack(xs), np.array(ys, dtype=np.int64), np.array(ids)

def to_arrays(ds):
    x = ds.imgs.astype(np.float32) / 255.0      # (N, H, W), values 0..1
    y = ds.labels.reshape(-1).astype(np.int64)
    return x, y

if DATA_SOURCE == "medmnist":
    # Download the official train / validation / test splits. Label 0 = normal, 1 = pneumonia.
    splits = {s: PneumoniaMNIST(split=s, download=True, size=IMG_SIZE) for s in ["train", "val", "test"]}
    data = {s: to_arrays(ds) for s, ds in splits.items()}
else:
    from sklearn.model_selection import GroupShuffleSplit
    root = find_kaggle_folder()
    print("Kaggle data folder:", root)
    with Timer("loading the Kaggle images"):
        xa, ya, ga = load_folder(root, "train", IMG_SIZE)       # official train split
        xv, yv, gv = load_folder(root, "val", IMG_SIZE)         # only 16 images, so we merge it into train
        xt, yt, _ = load_folder(root, "test", IMG_SIZE)         # official test split (624 images, never used for choices)
    x_all, y_all, g_all = np.concatenate([xa, xv]), np.concatenate([ya, yv]), np.concatenate([ga, gv])
    # Our own validation set (about 10%), split by PATIENT so the same child is never in both train and validation
    tr_idx, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED).split(x_all, y_all, g_all))
    f32 = lambda a: a.astype(np.float32) / 255.0
    data = {"train": (f32(x_all[tr_idx]), y_all[tr_idx]), "val": (f32(x_all[va_idx]), y_all[va_idx]),
            "test": (f32(xt), yt)}
    print(f"Patients in both train and validation: {len(set(g_all[tr_idx]) & set(g_all[va_idx]))} (should be 0)")
print("Data source:", DATA_SOURCE, "| image size:", IMG_SIZE)
for s, (x, y) in data.items():
    print(f"{s:5s}: {len(y):5d} images | normal={np.sum(y==0):4d} | pneumonia={np.sum(y==1):4d} "
          f"| pneumonia share={y.mean():.0%}")

In [ ]:
# Look at a few examples from each class
x_tr, y_tr = data["train"]
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for row, label in enumerate([0, 1]):
    idx = np.random.choice(np.where(y_tr == label)[0], 6, replace=False)
    for ax, i in zip(axes[row], idx):
        ax.imshow(x_tr[i], cmap="gray"); ax.axis("off")
    axes[row][0].set_title(["NORMAL", "PNEUMONIA"][label], loc="left", fontsize=12)
plt.tight_layout(); plt.show()

## 2. Baseline: always predict "pneumonia"

About three in four training images show pneumonia, so a "model" that labels **every** X-ray as pneumonia
gets a high accuracy score while being useless: it raises a false alarm for every healthy patient.
For that reason we always report **missed cases (false negatives)** and **false alarms (false positives)**
separately, not just accuracy.

In [ ]:
def report(name, y_true, y_pred, y_prob=None):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    r = {
        "model": name,
        "accuracy": (tp + tn) / len(y_true),
        "recall (pneumonia caught)": tp / (tp + fn) if tp + fn else float("nan"),
        "specificity (normals cleared)": tn / (tn + fp) if tn + fp else float("nan"),
        "missed cases (FN)": int(fn),
        "false alarms (FP)": int(fp),
        "AUC": roc_auc_score(y_true, y_prob) if y_prob is not None else float("nan"),
    }
    print(f"\n=== {name} ===")
    for k, v in r.items():
        if k != "model":
            print(f"  {k:30s} {v:.3f}" if isinstance(v, float) else f"  {k:30s} {v}")
    return r

results = []
x_te, y_te = data["test"]
results.append(report("Baseline: always pneumonia", y_te, np.ones_like(y_te)))

## 3. Model 1: a small CNN trained from scratch

In [ ]:
def make_loader(split, shuffle, batch_size=128, augment=False):
    x, y = data[split]
    ds = TensorDataset(torch.from_numpy(x).unsqueeze(1), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader("train", shuffle=True)
val_loader   = make_loader("val",   shuffle=False)
test_loader  = make_loader("test",  shuffle=False)

# The classes are imbalanced (about 3 pneumonia : 1 normal), so we weight the
# rarer "normal" class more in the loss. Otherwise the model learns to say "pneumonia" too often.
counts = np.bincount(data["train"][1])
class_weights = torch.tensor(len(data["train"][1]) / (2 * counts), dtype=torch.float32).to(device)
print("Class weights (normal, pneumonia):", class_weights.cpu().numpy().round(2))

def augment_batch(xb):
    # Small random changes so the model doesn't memorise exact pixels:
    # random horizontal shift/zoom via affine grid + brightness/contrast jitter.
    n = xb.size(0)
    theta = torch.zeros(n, 2, 3, device=xb.device)
    scale = 1 + (torch.rand(n, device=xb.device) - 0.5) * 0.2      # ±10% zoom
    theta[:, 0, 0] = scale; theta[:, 1, 1] = scale
    theta[:, :, 2] = (torch.rand(n, 2, device=xb.device) - 0.5) * 0.1  # ±5% shift
    grid = F.affine_grid(theta, xb.shape, align_corners=False)
    xb = F.grid_sample(xb, grid, padding_mode="border", align_corners=False)
    xb = xb * (1 + (torch.rand(n, 1, 1, 1, device=xb.device) - 0.5) * 0.2)
    return xb.clamp(0, 1)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                                 nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                                 nn.MaxPool2d(2))
        self.features = nn.Sequential(block(1, 32), block(32, 64), block(64, 128))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 2))
    def forward(self, x):
        return self.head(self.features(x))

@torch.no_grad()
def predict_probs(model, loader, prep=lambda x: x):
    model.eval()
    probs = []
    for xb, _ in loader:
        logits = model(prep(xb.to(device)))
        probs.append(torch.softmax(logits, 1)[:, 1].cpu())
    return torch.cat(probs).numpy()

def train(model, epochs, lr, prep=lambda x: x):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1))
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    best_auc, best_state = -1, None
    y_val = data["val"][1]
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); total = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(prep(augment_batch(xb))), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(yb)
        sched.step()
        val_auc = roc_auc_score(y_val, predict_probs(model, val_loader, prep))
        print(f"epoch {ep:2d} | train loss {total/len(train_loader.dataset):.4f} | val AUC {val_auc:.4f} | {time.time()-t0:.0f}s")
        if val_auc > best_auc:   # keep the best epoch on VALIDATION data (never test)
            best_auc = val_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f"Best val AUC: {best_auc:.4f}")
    return model

cnn = train(SmallCNN(), EPOCHS_CNN, lr=1e-3)

In [ ]:
cnn_test_probs = predict_probs(cnn, test_loader)
results.append(report("Small CNN (threshold 0.5)", y_te, (cnn_test_probs >= 0.5).astype(int), cnn_test_probs))

## 4. Model 2: fine-tune a pretrained ResNet-18

ResNet-18 was pretrained on ImageNet (everyday photos, not X-rays). We copy each grey X-ray into 3 colour
channels, enlarge it to 128 pixels and normalise it the way ImageNet images were, then fine-tune the whole
network. This step is much faster on a GPU.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def resnet_prep(xb):
    xb = F.interpolate(xb, size=128, mode="bilinear", align_corners=False).repeat(1, 3, 1, 1)
    return (xb - IMAGENET_MEAN.to(xb.device)) / IMAGENET_STD.to(xb.device)

resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
resnet.fc = nn.Linear(resnet.fc.in_features, 2)
resnet = train(resnet, EPOCHS_RESNET, lr=3e-4, prep=resnet_prep)

resnet_test_probs = predict_probs(resnet, test_loader, resnet_prep)
results.append(report("ResNet-18 (threshold 0.5)", y_te, (resnet_test_probs >= 0.5).astype(int), resnet_test_probs))

## 5. Choosing the threshold and the "unsure" band (our own addition)

A model outputs a probability, and **we** decide where to draw the line. In a clinic without a radiologist,
**a missed pneumonia is worse than a false alarm**: a false alarm costs a few minutes of a clinician's time,
while a missed case can delay treatment for a sick patient.

We make every choice on the **validation** set and only then look at the **test** set:
1. Pick the model with the higher validation AUC.
2. **Threshold:** the highest threshold that still catches at least **95%** of pneumonia cases on validation data.
3. **Unsure band:** below the threshold, scans that are still fairly close to it are marked UNSURE instead of
   "likely normal". We set the lower edge so that at least **99%** of validation pneumonia cases score above it.
   Only scans below that edge get "likely normal".

In [ ]:
x_val, y_val = data["val"]
val_probs = {"Small CNN": predict_probs(cnn, val_loader),
             "ResNet-18": predict_probs(resnet, val_loader, resnet_prep)}
for k, p in val_probs.items():
    print(f"{k:10s} val AUC = {roc_auc_score(y_val, p):.4f}")

best_name = max(val_probs, key=lambda k: roc_auc_score(y_val, val_probs[k]))
best_model = cnn if best_name == "Small CNN" else resnet
best_prep = (lambda x: x) if best_name == "Small CNN" else resnet_prep
p_val = val_probs[best_name]
p_test = cnn_test_probs if best_name == "Small CNN" else resnet_test_probs
print("Chosen model:", best_name)

def threshold_for_recall(probs, y, target):
    # Highest threshold t such that recall(probs >= t) >= target
    pos = np.sort(probs[y == 1])
    k = int(np.floor((1 - target) * len(pos)))   # how many positives we may miss
    return float(pos[k])

TARGET_RECALL = 0.95
SAFETY_RECALL = 0.99
THRESHOLD = threshold_for_recall(p_val, y_val, TARGET_RECALL)
UNSURE_LOW = min(threshold_for_recall(p_val, y_val, SAFETY_RECALL), THRESHOLD)
print(f"Threshold (≥{TARGET_RECALL:.0%} val recall): {THRESHOLD:.3f}")
print(f"Unsure band: {UNSURE_LOW:.4f} ≤ p < {THRESHOLD:.4f}")

In [ ]:
# Precision / recall trade-off on validation, with our chosen threshold marked
ts = np.linspace(0, 1, 201)
rec = [((p_val >= t) & (y_val == 1)).sum() / (y_val == 1).sum() for t in ts]
spec = [((p_val < t) & (y_val == 0)).sum() / (y_val == 0).sum() for t in ts]
plt.figure(figsize=(7, 4))
plt.plot(ts, rec, label="recall (pneumonia caught)")
plt.plot(ts, spec, label="specificity (normals cleared)")
plt.axvline(THRESHOLD, color="red", ls="--", label=f"chosen threshold {THRESHOLD:.2f}")
plt.axvspan(UNSURE_LOW, THRESHOLD, color="gold", alpha=0.25, label="unsure band")
plt.xlabel("threshold"); plt.title(f"{best_name}: validation set"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 6. Final test results

These numbers come from the **test set**, which played no part in training or in choosing the threshold.

In [ ]:
results.append(report(f"{best_name} (safety threshold {THRESHOLD:.2f})", y_te,
                      (p_test >= THRESHOLD).astype(int), p_test))

def triage(p):
    return np.where(p >= THRESHOLD, "URGENT", np.where(p >= UNSURE_LOW, "UNSURE", "LIKELY NORMAL"))

t_test = triage(p_test)
print("\n=== Triage outcome on the test set ===")
for label in ["URGENT", "UNSURE", "LIKELY NORMAL"]:
    m = t_test == label
    print(f"  {label:14s} {m.sum():4d} scans | actually pneumonia: {(y_te[m]==1).sum():4d} | actually normal: {(y_te[m]==0).sum():4d}")

missed_in_normal_pile = int(((t_test == "LIKELY NORMAL") & (y_te == 1)).sum())
print(f"\nPneumonia cases that ended up in the LIKELY NORMAL pile (the dangerous misses): {missed_in_normal_pile} "
      f"of {(y_te==1).sum()} ({missed_in_normal_pile/(y_te==1).sum():.1%})")
print(f"Share of scans a person must look at first (URGENT + UNSURE): {(t_test != 'LIKELY NORMAL').mean():.0%}")

In [ ]:
import pandas as pd
summary = pd.DataFrame(results).set_index("model").round(3)
summary

In [ ]:
cm = confusion_matrix(y_te, (p_test >= THRESHOLD).astype(int))
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=16,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks([0, 1]); ax.set_xticklabels(["pred normal", "pred pneumonia"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["true normal", "true pneumonia"])
ax.set_title(f"{best_name}, test set"); plt.tight_layout(); plt.show()

## 7. Where it fails

Below are the test scans the model got **most wrong**: real pneumonia cases with the lowest scores (missed
cases), and normal scans with the highest scores (false alarms). Look at them and write down in the write-up
**why** you think the model fails on them, for example faint or small patches of infection, poor image
contrast, a patient rotated in the image, or labels that may be wrong.

In [ ]:
fn_idx = np.where(y_te == 1)[0][np.argsort(p_test[y_te == 1])[:6]]      # lowest-scoring pneumonia
fp_idx = np.where(y_te == 0)[0][np.argsort(-p_test[y_te == 0])[:6]]     # highest-scoring normals
fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for ax, i in zip(axes[0], fn_idx):
    ax.imshow(x_te[i], cmap="gray"); ax.axis("off"); ax.set_title(f"#{i}  p={p_test[i]:.2f}\n→ {triage(p_test[i:i+1])[0]}", fontsize=9)
for ax, i in zip(axes[1], fp_idx):
    ax.imshow(x_te[i], cmap="gray"); ax.axis("off"); ax.set_title(f"#{i}  p={p_test[i]:.2f}\n→ {triage(p_test[i:i+1])[0]}", fontsize=9)
axes[0][0].text(-0.3, 0.5, "MISSED\n(true pneumonia)", transform=axes[0][0].transAxes, ha="right", va="center", fontsize=10, color="crimson")
axes[1][0].text(-0.3, 0.5, "FALSE ALARM\n(true normal)", transform=axes[1][0].transAxes, ha="right", va="center", fontsize=10, color="darkorange")
plt.tight_layout(); plt.show()

**A limit to state honestly:** PneumoniaMNIST comes from a single hospital in Guangzhou, China, and contains
X-rays of **children aged 1–5** (Kermany et al., 2018). The dataset includes no age, sex or scanner details, so we
**cannot check here whether the model works equally well for different groups**, and we should expect it to
perform worse on adult X-rays or scans from other machines. Testing on X-rays from a different source is the
most important next step.

## 8. Visual interpretability (Grad-CAM)

**Grad-CAM** colours the parts of the image that pushed the model towards "pneumonia". Red means strong
influence. We look at three groups: pneumonia the model caught, pneumonia it missed, and normal scans it
flagged by mistake.

**Shortcut check:** a model can score well by learning the wrong thing, for example text markers or the edges
of the film instead of the lungs. We measure what share of the heatmap falls in the outer 15% border of the
image. If the heat were spread evenly, about 51% would land there (that's the border's share of the area),
so a value well below 51% means the model mostly looks at the middle of the chest.

*Note: the heatmaps are coarse, because the network's last feature map is only 8×8 cells (or 16×16 for the
small CNN). Treat them as a rough guide, not proof.*

In [ ]:
with Timer("Section 8: Grad-CAM"):
    def cam_layer(model):
        # The last convolutional stage with enough spatial detail
        return model.layer3 if isinstance(model, torchvision.models.ResNet) else model.features[-1][5]

    def grad_cam(model, prep, x_raw):
        # x_raw: (1, 1, H, W) tensor with values 0..1. Returns (heatmap H×W in 0..1, pneumonia probability)
        store = {}
        def hook(module, inp, out):
            store["act"] = out
            out.register_hook(lambda g: store.__setitem__("grad", g))
        h = cam_layer(model).register_forward_hook(hook)
        model.eval(); model.zero_grad()
        with torch.enable_grad():
            logits = model(prep(x_raw.to(device)))
            logits[0, 1].backward()
        h.remove()
        weights = store["grad"].mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * store["act"]).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x_raw.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam / (cam.max() + 1e-8)).detach().cpu().numpy()
        return cam, torch.softmax(logits, 1)[0, 1].item()

    def overlay(gray, cam, alpha=0.45):
        rgb = np.repeat(gray[..., None], 3, axis=2)
        return np.clip((1 - alpha) * rgb + alpha * plt.cm.jet(cam)[..., :3], 0, 1)

    tp_idx = np.where((y_te == 1) & (p_test >= THRESHOLD))[0][:4]
    groups = [("CAUGHT (true pneumonia)", tp_idx, "green"),
              ("MISSED (true pneumonia)", fn_idx[:4], "crimson"),
              ("FALSE ALARM (true normal)", fp_idx[:4], "darkorange")]
    fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
    for row, (title, idxs, colour) in enumerate(groups):
        for k in range(4):
            a1, a2 = axes[row][2 * k], axes[row][2 * k + 1]
            a1.axis("off"); a2.axis("off")
            if k >= len(idxs):
                continue
            i = idxs[k]
            cam, p = grad_cam(best_model, best_prep, torch.from_numpy(x_te[i])[None, None])
            a1.imshow(x_te[i], cmap="gray"); a1.set_title(f"#{i} p={p:.2f}", fontsize=8)
            a2.imshow(overlay(x_te[i], cam)); a2.set_title("Grad-CAM", fontsize=8)
        axes[row][0].text(-0.15, 0.5, title, transform=axes[row][0].transAxes, ha="right", va="center",
                          fontsize=10, color=colour)
    plt.tight_layout(); plt.show()

    # Shortcut check: share of heat in the outer 15% border
    def border_share(cam, frac=0.15):
        h, w = cam.shape; b = int(round(h * frac))
        return 1 - cam[b:h - b, b:w - b].sum() / (cam.sum() + 1e-8)

    n_check = 30 if QUICK_RUN else 200
    check_idx = np.random.RandomState(0).choice(len(y_te), size=n_check, replace=False)
    shares = np.array([border_share(grad_cam(best_model, best_prep, torch.from_numpy(x_te[i])[None, None])[0])
                       for i in check_idx])
    area_share = 1 - (1 - 2 * 0.15) ** 2
    print(f"Average share of Grad-CAM heat in the image border: {shares.mean():.0%} "
          f"(evenly spread heat would give {area_share:.0%}) over {n_check} test X-rays")
    print(f"X-rays where more than half the heat sits in the border: {(shares > 0.5).sum()} of {n_check}")

**What to write about this:** do the heatmaps sit on the lungs? Do the missed cases show heat in the wrong
place, or almost none? If the border share is high, the model may be using shortcuts, which is worth saying
honestly in the write-up.

## 9. Edge-case and noise robustness

Real clinic X-rays are often worse than the clean dataset: photographed from a screen, compressed by a
messaging app, scanned at low resolution or slightly rotated. We change the **test** images in each of these
ways and measure missed cases and false alarms again, keeping the same threshold.

We also feed the model **edge cases** that aren't chest X-rays at all (blank images, random noise, an inverted
X-ray). A classifier always outputs *some* probability, even for nonsense, so we add an **input check**. It
compares an image's brightness, contrast and similarity to the average training X-ray with the range seen in
training, and refuses images outside that range ("cannot assess").

In [ ]:
import io
import torchvision.transforms.functional as TF
from PIL import Image

with Timer("Section 9: robustness"):
    def probs_for(x):
        ds = TensorDataset(torch.from_numpy(x.astype(np.float32)).unsqueeze(1), torch.zeros(len(x), dtype=torch.long))
        return predict_probs(best_model, DataLoader(ds, batch_size=128), best_prep)

    def to_t(x): return torch.from_numpy(x.astype(np.float32))[:, None]

    def jpeg(x, quality=20):
        out = []
        for im in x:
            buf = io.BytesIO()
            Image.fromarray((im * 255).astype(np.uint8)).save(buf, format="JPEG", quality=quality)
            out.append(np.asarray(Image.open(buf), dtype=np.float32) / 255)
        return np.stack(out)

    rng = np.random.RandomState(0)
    PERTURBATIONS = {
        "clean (no change)":      lambda x: x,
        "noise σ=0.05":           lambda x: np.clip(x + rng.normal(0, 0.05, x.shape), 0, 1),
        "noise σ=0.10":           lambda x: np.clip(x + rng.normal(0, 0.10, x.shape), 0, 1),
        "blur":                   lambda x: TF.gaussian_blur(to_t(x), kernel_size=5, sigma=1.5)[:, 0].numpy(),
        "darker (×0.7)":          lambda x: x * 0.7,
        "brighter (×1.3)":        lambda x: np.clip(x * 1.3, 0, 1),
        "low contrast":           lambda x: 0.5 * (x - x.mean((1, 2), keepdims=True)) + x.mean((1, 2), keepdims=True),
        "rotated 10°":            lambda x: TF.rotate(to_t(x), 10)[:, 0].numpy(),
        "low resolution (32 px)": lambda x: F.interpolate(F.interpolate(to_t(x), size=32, mode="area"),
                                                          size=x.shape[-1], mode="bilinear", align_corners=False)[:, 0].numpy(),
        "JPEG quality 20":        jpeg,
        "mirrored left-right":    lambda x: x[:, :, ::-1].copy(),
    }

    # ---- Input check, built from TRAINING images only ----
    MEAN_XRAY = data["train"][0].mean(0)
    def image_stats(x):
        m = x.mean((1, 2)); s = x.std((1, 2))
        a = x - m[:, None, None]; b = MEAN_XRAY - MEAN_XRAY.mean()
        corr = (a * b).sum((1, 2)) / (np.sqrt((a ** 2).sum((1, 2)) * (b ** 2).sum()) + 1e-8)
        return m, s, corr
    tm, ts, tc = image_stats(data["train"][0])
    LIMITS = {"mean": np.percentile(tm, [0.5, 99.5]), "std": np.percentile(ts, [0.5, 99.5]),
              "corr": np.percentile(tc, 0.5)}

    def input_check(x):
        # x: (N, H, W) in 0..1. Returns a list of reasons; an empty string means the image looks like a training X-ray
        m, s, c = image_stats(x.astype(np.float32))
        reasons = []
        for mi, si, ci in zip(m, s, c):
            r = []
            if not (LIMITS["mean"][0] <= mi <= LIMITS["mean"][1]): r.append("unusual brightness")
            if not (LIMITS["std"][0] <= si <= LIMITS["std"][1]):   r.append("unusual contrast")
            if not (ci >= LIMITS["corr"]):                         r.append("doesn't look like a chest X-ray")
            reasons.append(", ".join(r))
        return reasons

    rows = []
    for name, fn in PERTURBATIONS.items():
        xp = fn(x_te).astype(np.float32)
        p = probs_for(xp)
        tn, fp, fn_, tp = confusion_matrix(y_te, (p >= THRESHOLD).astype(int), labels=[0, 1]).ravel()
        rejected = np.mean([r != "" for r in input_check(xp)])
        rows.append({"test condition": name, "recall (pneumonia caught)": tp / (tp + fn_),
                     "missed cases (FN)": int(fn_), "false alarms (FP)": int(fp),
                     "false alarm rate": fp / (fp + tn), "AUC": roc_auc_score(y_te, p),
                     "refused by input check": rejected})

In [ ]:
robustness = pd.DataFrame(rows).set_index("test condition").round(3)
display(robustness)

fig, ax = plt.subplots(figsize=(11, 4))
xs = np.arange(len(robustness))
ax.bar(xs - 0.2, robustness["recall (pneumonia caught)"], 0.4, label="recall (pneumonia caught)", color="#0f766e")
ax.bar(xs + 0.2, robustness["false alarm rate"], 0.4, label="false alarm rate", color="#f59e0b")
ax.set_xticks(xs); ax.set_xticklabels(robustness.index, rotation=30, ha="right")
ax.axhline(robustness.loc["clean (no change)", "recall (pneumonia caught)"], color="#0f766e", ls=":", lw=1)
ax.set_ylim(0, 1.05); ax.legend(); ax.set_title(f"{best_name}: robustness on the test set (same threshold)")
plt.tight_layout(); plt.show()

In [ ]:
with Timer("Section 9b: edge cases"):
    s = IMG_SIZE
    yy, xx = np.mgrid[0:s, 0:s]
    sample_xray = x_te[np.where(y_te == 1)[0][0]]
    EDGE_CASES = {
        "black image":             np.zeros((s, s)),
        "white image":             np.ones((s, s)),
        "flat grey":               np.full((s, s), 0.5),
        "random noise":            np.random.RandomState(1).rand(s, s),
        "checkerboard":            (((yy // 8) + (xx // 8)) % 2).astype(float),
        "inverted X-ray":          1 - sample_xray,
        "X-ray rotated 90°":       np.rot90(sample_xray).copy(),
        "X-ray + heavy noise":     np.clip(sample_xray + np.random.RandomState(2).normal(0, 0.3, (s, s)), 0, 1),
        "real X-ray (for comparison)": sample_xray,
    }
    ex = np.stack(list(EDGE_CASES.values())).astype(np.float32)
    ep = probs_for(ex)
    ereasons = input_check(ex)
    fig, axes = plt.subplots(1, len(EDGE_CASES), figsize=(18, 3))
    for ax, (name, im), p, r in zip(axes, EDGE_CASES.items(), ep, ereasons):
        ax.imshow(im, cmap="gray", vmin=0, vmax=1); ax.axis("off")
        verdict = "REFUSED" if r else triage(np.array([p]))[0]
        ax.set_title(f"{name}\np={p:.2f} → {verdict}", fontsize=8, color="crimson" if r else "black")
    plt.tight_layout(); plt.show()
    edge_table = pd.DataFrame({"model probability": ep.round(3),
                               "model alone would say": triage(ep),
                               "input check": [r or "OK" for r in ereasons]}, index=list(EDGE_CASES))
    display(edge_table)
    clean_refused = np.mean([r != "" for r in input_check(x_te)])
    print(f"Real test X-rays wrongly refused by the input check: {clean_refused:.1%}")

**What to look for:** which conditions hurt recall the most? A big drop under JPEG or low resolution matters,
because that's how X-rays often travel between clinics (photos sent through messaging apps). Without the
input check, does the model call a black image or random noise "URGENT" or "LIKELY NORMAL" with confidence?
That's a strong argument for the check.

## 10. Multi-modal metadata: vital signs and danger signs

An X-ray alone doesn't tell the whole story. Nurses also measure **oxygen saturation (SpO₂)**, count the
**breathing rate** and look for **danger signs**. We combine these with the image result using simple rules
based on WHO guidance for children under 5 (WHO, *Revised classification and treatment of pneumonia in
children at health facilities*, 2014; IMCI chart booklet):

| Rule | Effect |
|---|---|
| SpO₂ below 90%, or any general danger sign (can't drink or breastfeed, vomits everything, convulsions, lethargic or unconscious) | **URGENT, clinical emergency**, whatever the X-ray says |
| Fast breathing for age: ≥60/min under 2 months, ≥50/min at 2–11 months, ≥40/min at 12–59 months | Raise urgency one level |
| Temperature ≥ 38 °C | Mentioned in the explanation, no change |
| Age outside 1–5 years | Warning: the model was trained only on this age range |

**Rules can only raise urgency, never lower it.** **Honest limit:** PneumoniaMNIST has no vital signs, so we
**could not test** whether these rules improve accuracy. They're fixed rules taken from guidance, not learned
from data. A real study would need X-rays paired with vital signs.

In [ ]:
LEVELS = ["LIKELY NORMAL", "UNSURE", "URGENT"]

def clinical_rules(image_label, age_months=None, resp_rate=None, spo2=None, temp_c=None, danger_signs=False):
    # Returns (final_label, emergency: bool, list of reasons shown to the nurse)
    level = LEVELS.index(image_label)
    reasons, emergency = [], False
    if spo2 is not None and spo2 > 0 and spo2 < 90:
        emergency = True; reasons.append(f"oxygen saturation {spo2:.0f}% is below 90%")
    if danger_signs:
        emergency = True; reasons.append("a general danger sign was reported")
    if resp_rate and age_months is not None and age_months < 60:
        limit = 60 if age_months < 2 else 50 if age_months < 12 else 40
        if resp_rate >= limit:
            level = min(level + 1, 2); reasons.append(f"fast breathing for age ({resp_rate:.0f}/min, limit {limit})")
    if temp_c and temp_c >= 38:
        reasons.append(f"fever ({temp_c:.1f} °C)")
    if age_months is not None and age_months > 0 and not (12 <= age_months < 72):
        reasons.append("age is outside the 1–5 years the model was trained on, so the image result is less reliable")
    if emergency:
        level = 2
    return LEVELS[level], emergency, reasons

examples = [
    ("LIKELY NORMAL", dict(age_months=30, resp_rate=30, spo2=97, temp_c=37.0)),
    ("LIKELY NORMAL", dict(age_months=30, resp_rate=46, spo2=96, temp_c=38.6)),
    ("UNSURE",        dict(age_months=8,  resp_rate=55, spo2=95, temp_c=38.2)),
    ("LIKELY NORMAL", dict(age_months=24, resp_rate=38, spo2=87, temp_c=37.5)),
    ("UNSURE",        dict(age_months=40, resp_rate=30, spo2=98, temp_c=36.8, danger_signs=True)),
    ("URGENT",        dict(age_months=120, resp_rate=28, spo2=97, temp_c=37.2)),
]
pd.DataFrame([{"image says": lab, **kw, "final": clinical_rules(lab, **kw)[0] + (" (EMERGENCY)" if clinical_rules(lab, **kw)[1] else ""),
               "why": "; ".join(clinical_rules(lab, **kw)[2]) or "no change"} for lab, kw in examples])

## 11. Triage queue simulation

Does sorting the queue actually help? We simulate many clinic days. Each morning a batch of X-rays arrives
(drawn from the **test set**, so the model never saw them), and one clinician reads them one by one. We compare:

- **First come, first served:** read in arrival order.
- **AI-ordered queue:** read in order of the model's pneumonia probability, highest first. This gives the same
  order as URGENT → UNSURE → LIKELY NORMAL.

We measure how long **pneumonia patients** wait for their X-ray to be read. We also report the wait for normal
patients, who pay the price by waiting longer. You can change the clinic settings in the demo app's
**Queue simulator** tab.

*Assumptions:* all X-rays arrive at once, every read takes the same time, and pneumonia makes up the share of
the batch that you set (the test set has 62%, which is higher than most clinics see).

In [ ]:
POLICIES = ["First come, first served", "AI-ordered queue"]

def simulate_queue(n_scans=40, minutes_per_read=6, prevalence=0.3, n_days=300, seed=0):
    rng = np.random.RandomState(seed)
    pos, neg = np.where(y_te == 1)[0], np.where(y_te == 0)[0]
    waits = {pol: {"pneumonia": [], "normal": []} for pol in POLICIES}
    for _ in range(n_days):
        is_pos = rng.rand(n_scans) < prevalence
        idx = np.where(is_pos, rng.choice(pos, n_scans), rng.choice(neg, n_scans))
        p = p_test[idx]
        order = {"First come, first served": np.arange(n_scans),
                 "AI-ordered queue": np.argsort(-p, kind="stable")}
        for pol, o in order.items():
            finish = np.empty(n_scans); finish[o] = (np.arange(n_scans) + 1) * minutes_per_read
            waits[pol]["pneumonia"] += list(finish[is_pos]); waits[pol]["normal"] += list(finish[~is_pos])
    summary = []
    for pol in POLICIES:
        wp, wn = np.array(waits[pol]["pneumonia"]), np.array(waits[pol]["normal"])
        summary.append({"policy": pol,
                        "pneumonia: average wait (min)": wp.mean() if len(wp) else np.nan,
                        "pneumonia: read within 1 hour": (wp <= 60).mean() if len(wp) else np.nan,
                        "pneumonia: 95% read within (min)": np.percentile(wp, 95) if len(wp) else np.nan,
                        "normal: average wait (min)": wn.mean() if len(wn) else np.nan})
    return pd.DataFrame(summary).set_index("policy").round(2), waits

def queue_plot(waits, minutes_per_read, n_scans):
    fig, ax = plt.subplots(figsize=(7, 4))
    t = np.arange(0, n_scans * minutes_per_read + 1)
    for pol, colour in zip(POLICIES, ["#64748b", "#0f766e"]):
        w = np.sort(waits[pol]["pneumonia"])
        if len(w):
            ax.plot(t, np.searchsorted(w, t, side="right") / len(w), label=pol, color=colour, lw=2.5)
    ax.axvline(60, color="crimson", ls=":", lw=1); ax.text(61, 0.03, "1 hour", color="crimson")
    ax.set_xlabel("minutes after the batch arrives"); ax.set_ylabel("share of pneumonia X-rays read")
    ax.set_title("How fast pneumonia cases get read"); ax.grid(alpha=.3); ax.legend()
    plt.tight_layout()
    return fig

with Timer("Section 11: queue simulation"):
    queue_summary, queue_waits = simulate_queue(n_days=50 if QUICK_RUN else 300)
    display(queue_summary)
    queue_plot(queue_waits, 6, 40); plt.show()

## 12. Inference latency and edge deployment

Could this run on a cheap computer in a clinic with no GPU and unreliable internet? We measure:

- **Latency:** milliseconds to analyse **one** X-ray. We report the median and the slowest 5% (p95).
- **Setups:** Colab GPU; Colab CPU with all cores and with **1 core** (closer to a low-end PC); and
  **ONNX Runtime**, a lightweight engine for running models without PyTorch. We also try **int8 quantisation**,
  which stores the weights 4× smaller, and check whether it changes the results.
- **Size:** how many MB the model takes.

*Honest limit:* Colab's CPU is not a real clinic computer. A Raspberry Pi or old laptop will be slower, so
these numbers show relative differences, and the tool should be benchmarked on the actual device.

In [ ]:
import copy

class Deployable(nn.Module):
    # Wraps preprocessing + model + softmax so the exported model takes a raw 0..1 X-ray and returns P(pneumonia)
    def __init__(self, model, prep):
        super().__init__(); self.model, self.prep = model, prep
    def forward(self, x):
        return torch.softmax(self.model(self.prep(x)), 1)[:, 1]

def bench(fn, n=50, warmup=5):
    for _ in range(warmup): fn()
    times = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times)), float(np.percentile(times, 95))

def export_onnx(module, path):
    dummy = torch.zeros(1, 1, IMG_SIZE, IMG_SIZE)
    kw = dict(input_names=["xray"], output_names=["p_pneumonia"],
              dynamic_axes={"xray": {0: "batch"}, "p_pneumonia": {0: "batch"}}, opset_version=17)
    try:
        torch.onnx.export(module, dummy, path, dynamo=False, **kw)   # classic exporter (newer PyTorch)
    except TypeError:
        torch.onnx.export(module, dummy, path, **kw)                 # older PyTorch has no 'dynamo' argument

N_RUNS = 10 if QUICK_RUN else 100
one = torch.from_numpy(x_te[:1]).unsqueeze(1)
lat_rows, onnx_paths = [], {}
default_threads = torch.get_num_threads()

with Timer("Section 12: latency benchmarks"):
    for name, model, prep in [("Small CNN", cnn, lambda x: x), ("ResNet-18", resnet, resnet_prep)]:
        size_mb = sum(p.numel() for p in model.parameters()) * 4 / 1e6
        cpu_mod = Deployable(copy.deepcopy(model).cpu(), prep).eval()
        def add(setup, ms, mb=size_mb): lat_rows.append({"model": name, "setup": setup, "median ms": ms[0], "p95 ms": ms[1], "size MB": mb})

        if torch.cuda.is_available():
            gpu_mod = Deployable(model, prep).eval()
            g = one.to("cuda")
            def run_gpu():
                with torch.no_grad(): gpu_mod(g)
                torch.cuda.synchronize()
            add("PyTorch, GPU", bench(run_gpu, N_RUNS))
        with torch.no_grad():
            add(f"PyTorch, CPU ({default_threads} cores)", bench(lambda: cpu_mod(one), N_RUNS))
            torch.set_num_threads(1)
            add("PyTorch, CPU (1 core)", bench(lambda: cpu_mod(one), N_RUNS))
            torch.set_num_threads(default_threads)

        try:
            import onnxruntime as ort
            path = f"{name.replace(' ', '_').replace('-', '')}.onnx"
            export_onnx(cpu_mod, path); onnx_paths[name] = path
            so = ort.SessionOptions(); so.intra_op_num_threads = 1
            sess = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])
            x_np = one.numpy()
            add("ONNX Runtime, CPU (1 core)", bench(lambda: sess.run(None, {"xray": x_np}), N_RUNS),
                os.path.getsize(path) / 1e6)
            with torch.no_grad():
                diff = np.abs(sess.run(None, {"xray": x_te[:64, None]})[0] - cpu_mod(torch.from_numpy(x_te[:64, None])).numpy()).max()
            print(f"{name}: ONNX vs PyTorch, largest difference in probability = {diff:.2e}")
            try:
                from onnxruntime.quantization import quantize_dynamic, QuantType
                qpath = path.replace(".onnx", "_int8.onnx")
                quantize_dynamic(path, qpath, weight_type=QuantType.QUInt8)   # unsigned int8: the version ONNX Runtime can run for conv layers on CPU
                qsess = ort.InferenceSession(qpath, so, providers=["CPUExecutionProvider"])
                add("ONNX Runtime int8, CPU (1 core)", bench(lambda: qsess.run(None, {"xray": x_np}), N_RUNS),
                    os.path.getsize(qpath) / 1e6)
                qp = np.concatenate([qsess.run(None, {"xray": x_te[i:i + 64, None]})[0] for i in range(0, len(x_te), 64)])
                tn, fp, fn_, tp = confusion_matrix(y_te, (qp >= THRESHOLD).astype(int), labels=[0, 1]).ravel()
                print(f"{name} int8 on the test set: recall {tp/(tp+fn_):.3f}, false alarms {fp}, AUC {roc_auc_score(y_te, qp):.3f}")
            except Exception as e:
                print(f"{name}: int8 quantisation skipped ({type(e).__name__}: {str(e)[:120]})")
        except Exception as e:
            print(f"{name}: ONNX benchmark skipped ({type(e).__name__}: {str(e)[:120]})")

latency = pd.DataFrame(lat_rows).set_index(["model", "setup"]).round(2)
display(latency)
print(f"\nThe chosen model ({best_name}) needs about "
      f"{latency.loc[(best_name, 'PyTorch, CPU (1 core)'), 'median ms']:.0f} ms per X-ray on a single CPU core, "
      f"so roughly {1000 / max(latency.loc[(best_name, 'PyTorch, CPU (1 core)'), 'median ms'], 1e-3):.0f} X-rays per second.")

## 13. Demo app

Upload a chest X-ray (PNG or JPG) to get a triage label. `share=True` gives a public link you can use in your demo video.

**Voice:** the app uses the web browser's own speech features. It reads the result and chat answers aloud, and the 🎤 button lets you ask a question by speaking. Use **Chrome or Edge** and open the public `gradio.live` link, since voice input needs a secure (https) page and microphone permission. Note that Chrome turns speech into text using Google's servers.

In [ ]:
from PIL import Image

def preprocess_upload(img):
    img = Image.fromarray(img) if isinstance(img, np.ndarray) else img
    img = img.convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    x = torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0)[None, None]
    return x

@torch.no_grad()
def screen(img):
    if img is None:
        return "Please upload an X-ray image.", {}
    best_model.eval()
    p = torch.softmax(best_model(best_prep(preprocess_upload(img).to(device))), 1)[0, 1].item()
    label = triage(np.array([p]))[0]
    advice = {
        "URGENT": "🔴 URGENT: signs consistent with pneumonia. Move this scan to the top of the reading queue.",
        "UNSURE": "🟡 UNSURE: the model is not confident. A clinician should review this scan soon.",
        "LIKELY NORMAL": "🟢 LIKELY NORMAL: no strong pneumonia signal. Keep it in the normal reading queue.",
    }[label]
    text = (f"{advice}\n\nPneumonia probability: {p:.1%} (urgent threshold {THRESHOLD:.1%}, "
            f"unsure from {UNSURE_LOW:.2%})\n\n⚠️ Prototype only, not medical advice. "
            f"Trained on paediatric X-rays from one hospital; a clinician must read every scan.")
    return text, {"pneumonia": p, "normal": 1 - p}

# Save a few test images so you can try the demo straight away
os.makedirs("examples", exist_ok=True)
example_paths = []
for i in list(fn_idx[:1]) + list(fp_idx[:1]) + list(np.where(y_te == 1)[0][:1]) + list(np.where(y_te == 0)[0][:1]):
    path = f"examples/test_{i}_{'pneumonia' if y_te[i] else 'normal'}.png"
    Image.fromarray((x_te[i] * 255).astype(np.uint8)).save(path)
    example_paths.append(path)

# Check the demo function gives the same answer as the notebook on one test image
print(screen(np.asarray(Image.open(example_paths[2])))[0])

### Plain-language explanations with Gemini (optional)

Our trained model makes the triage decision. Gemini **only explains** that decision in plain words for the
clinic nurse. It receives **only the numbers** (probability, label, thresholds and our test-set error rates),
**never the X-ray image**, and it is told not to change the label or give a diagnosis.

**Keep your API key out of this notebook.** Judges get a public link to it. Instead:
1. Click the **🔑 key icon** in Colab's left sidebar (Secrets).
2. Click **Add new secret**. Name: `GEMINI_API_KEY`. Value: your key.
3. Turn on **Notebook access** for it.

**Or, quicker:** if there's no secret, this cell shows an input box. Paste your key there and press Enter.
It's hidden while you type and is **not** saved in the notebook. You'll need to paste it again each time
you restart Colab.

If you skip both, the app still works and shows a fixed explanation instead.

In [ ]:
# ⚠️ This key is visible to ANYONE who can open this notebook. Delete it (set it to "") before sharing
#    the notebook publicly, or delete the key in Google AI Studio after the hackathon.
GEMINI_API_KEY_IN_NOTEBOOK = "AQ.Ab8RN6J_-kCKp0TIdJTamSnXGFVRSYtcelLAH4HF22sw4KZoFw"

# Google retires model names over time (gemini-2.5-flash is no longer available to new keys).
# We try these in order and use the first one that answers. Add newer names at the front if needed:
# see https://ai.google.dev/gemini-api/docs/models
GEMINI_MODELS_TO_TRY = ["gemini-3.8-flash", "gemini-flash-latest", "gemini-3.5-flash",
                        "gemini-3-flash-preview", "gemini-flash-lite-latest", "gemini-2.5-flash"]
GEMINI_MODEL = None

def gemini_error(e):
    # A readable error: type, HTTP code and Google's message
    code, msg = getattr(e, "code", ""), getattr(e, "message", None) or str(e)
    return f"{type(e).__name__} {code}: {str(msg)[:200]}".strip()

gemini_client = None
try:
    from google.colab import userdata
    _key = userdata.get("GEMINI_API_KEY")
except Exception:
    _key = os.environ.get("GEMINI_API_KEY")
if not _key:
    _key = GEMINI_API_KEY_IN_NOTEBOOK
if not _key and not QUICK_RUN:
    # No secret found: ask for the key in a hidden box. It's used for this session only and NOT saved in the notebook.
    from getpass import getpass
    _key = getpass("Paste your Gemini API key and press Enter (it won't be saved), or just press Enter to skip: ").strip()
if _key:
    from google import genai
    from google.genai import types
    try:
        gemini_client = genai.Client(api_key=_key, http_options=types.HttpOptions(timeout=60_000))  # 60 s limit
    except Exception:
        gemini_client = genai.Client(api_key=_key)
    _errors = []
    for _m in GEMINI_MODELS_TO_TRY:
        try:
            _r = gemini_client.models.generate_content(model=_m, contents="Reply with the word OK.")
            if _r.text:
                GEMINI_MODEL = _m
                break
        except Exception as e:
            _errors.append(f"  {_m}: {gemini_error(e)}")
    if GEMINI_MODEL is None:
        gemini_client = None
        print("Gemini could not be reached with any model. Errors:\n" + "\n".join(_errors))
if gemini_client:
    print(f"Gemini explanations: ON (model: {GEMINI_MODEL})")
else:
    print("Gemini explanations: OFF. The app will show fixed explanations instead.")

# Test-set error rates of the chosen model at our threshold, so the explanation can be honest about them
_cm = confusion_matrix(y_te, (p_test >= THRESHOLD).astype(int), labels=[0, 1]).ravel()
TEST_STATS = {"recall": _cm[3] / (_cm[3] + _cm[2]), "false_alarm_rate": _cm[1] / (_cm[1] + _cm[0])}

GEMINI_INSTRUCTIONS = (
    "You explain the output of a pneumonia triage prototype to a nurse at a clinic with no radiologist. "
    "A separate image model already produced the label; you never see the image and must not change the label, "
    "guess a diagnosis, or suggest medicines or doses. Write 3 to 4 short, plain sentences: what the label means, "
    "how sure the model is, how often it is wrong (use the error rates given), and what the nurse should do next "
    "(always: a clinician must read the X-ray). Do not use headings or bullet points."
)

FALLBACK_TEXT = {
    "URGENT": "The model found a strong pneumonia signal, so this scan should be read first. It can still be wrong, so a clinician must read the X-ray before any decision.",
    "UNSURE": "The model is not confident either way. Treat this scan as a priority and have a clinician read it soon.",
    "LIKELY NORMAL": "The model found no strong pneumonia signal. The scan stays in the normal queue and a clinician still reads it. If the patient looks unwell, don't wait.",
}

def result_facts(p, label, clinical_notes="", image_label=None):
    # The only information Gemini ever receives: numbers and labels, never the image
    image_label = image_label or label
    return (f"Final label: {label}. The X-ray model alone said: {image_label}"
            f"{' (the final label was raised by the nurse observations, not by the X-ray)' if image_label != label else ''}. "
            f"{('Nurse observations that affected or accompany the label: ' + clinical_notes + '. ') if clinical_notes else ''}"
            f"Image model pneumonia probability: {p:.1%}. URGENT if probability >= {THRESHOLD:.1%}; "
            f"UNSURE if between {UNSURE_LOW:.2%} and {THRESHOLD:.1%}; otherwise LIKELY NORMAL. "
            f"On held-out test X-rays this model caught {TEST_STATS['recall']:.0%} of pneumonia cases and "
            f"wrongly flagged {TEST_STATS['false_alarm_rate']:.0%} of normal scans. "
            f"Training data: chest X-rays of children aged 1-5 from one hospital.")

def explain(p, label, clinical_notes="", image_label=None):
    image_label = image_label or label
    if gemini_client is None:
        if image_label != label:   # urgency came from the patient's observations, not the X-ray
            return (f"The X-ray alone looked {image_label.lower()} (pneumonia probability {p:.1%}), but the patient's "
                    f"observations listed above raised the urgency to {label}. A clinician should assess the patient "
                    f"and read the X-ray.")
        return FALLBACK_TEXT[label]
    try:
        resp = gemini_client.models.generate_content(
            model=GEMINI_MODEL, contents=result_facts(p, label, clinical_notes, image_label),
            config=types.GenerateContentConfig(system_instruction=GEMINI_INSTRUCTIONS, temperature=0.3),
        )
        return (resp.text or "").strip() or FALLBACK_TEXT[label]
    except Exception as e:
        return FALLBACK_TEXT[label] + f"\n\n(Gemini unavailable: {gemini_error(e)})"

# ---- Follow-up chat: the nurse can ask questions about the CURRENT result ----
CHAT_INSTRUCTIONS = (
    "You answer a clinic nurse's follow-up questions about one result from a pneumonia triage prototype. "
    "The facts about the result are in the first message. Use only those facts and general, well-established "
    "health information. Never change the label, never diagnose, never suggest medicines or doses: for treatment "
    "questions say a clinician must decide. If the result is a clinical emergency, remind them to assess the patient now. "
    "If you don't know, say so. Answer in 2 to 4 short, plain sentences without headings or bullet points. "
    "Remember this is a prototype and not medical advice."
)

def chat_reply(message, history, facts):
    # history: list of {"role": "user"/"assistant", "content": str}
    if not facts:
        return "Please check an X-ray first, then ask me about the result."
    if gemini_client is None:
        return ("Gemini isn't connected (no GEMINI_API_KEY secret), so I can't answer free questions. "
                "Please ask a clinician, and see 'How reliable is this?' below.")
    contents = [types.Content(role="user", parts=[types.Part(text="Facts about the result: " + facts)]),
                types.Content(role="model", parts=[types.Part(text="Understood. I'll only use these facts.")])]
    for m in history:
        contents.append(types.Content(role="model" if m["role"] == "assistant" else "user",
                                      parts=[types.Part(text=str(m["content"]))]))
    contents.append(types.Content(role="user", parts=[types.Part(text=message)]))
    try:
        resp = gemini_client.models.generate_content(
            model=GEMINI_MODEL, contents=contents,
            config=types.GenerateContentConfig(system_instruction=CHAT_INSTRUCTIONS, temperature=0.3))
        return (resp.text or "").strip() or "Sorry, I couldn't produce an answer. Please ask a clinician."
    except Exception as e:
        return f"Gemini is unavailable right now ({gemini_error(e)}). Please ask a clinician."

# ---- Voice: the BROWSER speaks and listens (Web Speech API), so no extra package is needed ----
import re
def speak_div(text):
    # A hidden block holding the text to read aloud; the page's JavaScript reads it with the browser's own voice
    clean = re.sub(r"[*_#`>]|\(Gemini unavailable.*?\)", "", text or "").strip()
    return f'<div class="speak" style="display:none;height:0;margin:0;padding:0;border:0">{html.escape(clean)}</div>' if clean else ""

print(explain(0.9, "URGENT"))

In [ ]:
import html
CARD_STYLE = {
    "URGENT":        ("#b91c1c", "#fef2f2", "🔴", "URGENT", "Read this X-ray first"),
    "UNSURE":        ("#a16207", "#fefce8", "🟡", "UNSURE", "A clinician should review this soon"),
    "LIKELY NORMAL": ("#15803d", "#f0fdf4", "🟢", "LIKELY NORMAL", "Keep in the normal reading queue"),
}

def result_card(p, label, emergency=False, reasons=(), ms=None, explanation=""):
    # ONE box: the patient's condition at the top, then the reasons underneath
    color, bg, icon, title, action = CARD_STYLE[label]
    if emergency:
        action = "CLINICAL EMERGENCY: assess the patient now, don't wait for the X-ray to be read"
    reason_html = "".join(f"<li>{html.escape(r)}</li>" for r in reasons)
    expl_html = html.escape(explanation).replace("\n", "<br>")
    return f'''
    <div style="border:2px solid {color};background:{bg};border-radius:14px;padding:18px 20px;color:#111">
      <div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;color:#555">Condition of the patient</div>
      <div style="font-size:26px;font-weight:700;color:{color};margin-top:2px">{icon} {title}</div>
      <div style="font-size:16px;margin-top:4px">{action}</div>
      <div style="margin-top:14px;font-size:14px">Pneumonia probability from the X-ray: <b>{p:.1%}</b></div>
      <div style="position:relative;height:14px;background:#e5e7eb;border-radius:7px;margin-top:6px;overflow:hidden">
        <div style="position:absolute;left:0;top:0;bottom:0;width:{p*100:.1f}%;background:{color}"></div>
        <div style="position:absolute;left:{THRESHOLD*100:.1f}%;top:-2px;bottom:-2px;width:2px;background:#111"></div>
      </div>
      <div style="font-size:12px;color:#444;margin-top:4px">Black line = urgent threshold ({THRESHOLD:.1%})
        {f" · analysed in {ms:.0f} ms" if ms is not None else ""}</div>
      <div style="height:0;border-top:1px solid {color}55;margin:14px 0 10px"></div>
      <div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;color:#555">Why</div>
      {f'<ul style="margin:6px 0 0 18px;font-size:14px">{reason_html}</ul>' if reasons else ''}
      <div style="font-size:15px;line-height:1.5;margin-top:6px">{expl_html}</div>
    </div>'''

REFUSED_CARD = '''
<div style="border:2px solid #475569;background:#f1f5f9;border-radius:14px;padding:18px 20px;color:#111">
  <div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;color:#555">Condition of the patient</div>
  <div style="font-size:24px;font-weight:700;color:#334155;margin-top:2px">⚪ CANNOT ASSESS</div>
  <div style="height:0;border-top:1px solid #47556955;margin:14px 0 10px"></div>
  <div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;color:#555">Why</div>
  <div style="font-size:15px;margin-top:6px">This image doesn't look like the chest X-rays the model was trained on ({why}).
  Check that it's a frontal chest X-ray, not blank or inverted, and send it for normal reading.</div>
</div>'''

def run_app(img, age_months, resp_rate, spo2, temp_c, danger_signs):
    # Returns (card HTML, heatmap image, text to read aloud)
    run_app.last_facts = ""
    if img is None:
        return "", None, ""
    t0 = time.perf_counter()
    x = preprocess_upload(img)
    why = input_check(x[0].numpy())[0]
    if why:
        spoken = f"Cannot assess. This image doesn't look like a chest X-ray the model knows: {why}. Please send it for normal reading."
        return REFUSED_CARD.format(why=html.escape(why)) + speak_div(spoken), None, spoken
    cam, p = grad_cam(best_model, best_prep, x)
    ms = (time.perf_counter() - t0) * 1000
    image_label = triage(np.array([p]))[0]
    label, emergency, reasons = clinical_rules(image_label, age_months or None, resp_rate or None,
                                               spo2 or None, temp_c or None, bool(danger_signs))
    if label != image_label:
        reasons = [f"The X-ray alone said {image_label}; raised because of the observations below"] + reasons
    gray = x[0, 0].numpy()
    heat = Image.fromarray((overlay(gray, cam) * 255).astype(np.uint8)).resize((256, 256), Image.BILINEAR)
    notes = "; ".join(reasons) + ("; clinical emergency" if emergency else "")
    explanation = explain(p, label, notes, image_label)
    run_app.last_facts = result_facts(p, label, notes, image_label)   # remembered for the follow-up chat
    _, _, _, title, action = CARD_STYLE[label]
    if emergency:
        action = "Clinical emergency. Assess the patient now"
    spoken = f"Condition of the patient: {title}. {action}. {explanation}"
    return result_card(p, label, emergency, reasons, ms, explanation) + speak_div(spoken), heat, spoken

def check_xray(img, age_months, resp_rate, spo2, temp_c, danger_signs):
    # Used by the app: shows the condition box and clears the chat (the browser then reads the box aloud)
    card, heat, spoken = run_app(img, age_months, resp_rate, spo2, temp_c, danger_signs)
    return card, heat, [], run_app.last_facts

def ask(message, history, facts):
    # Returns (chat history, hidden answer for the browser to read aloud, cleared question box)
    if not message or not message.strip():
        return history, "", ""
    answer = chat_reply(message, history, facts)
    history = history + [{"role": "user", "content": message}, {"role": "assistant", "content": answer}]
    return history, speak_div(answer), ""

def run_queue(n_scans, minutes_per_read, prevalence_pct):
    table, waits = simulate_queue(int(n_scans), float(minutes_per_read), prevalence_pct / 100, n_days=200)
    fifo, ai = table.loc[POLICIES[0]], table.loc[POLICIES[1]]
    text = (f"**Pneumonia X-rays read within 1 hour:** {fifo['pneumonia: read within 1 hour']:.0%} first-come-first-served "
            f"→ **{ai['pneumonia: read within 1 hour']:.0%} with the AI-ordered queue**.\n\n"
            f"**Average wait for pneumonia patients:** {fifo['pneumonia: average wait (min)']:.0f} min → "
            f"**{ai['pneumonia: average wait (min)']:.0f} min**. "
            f"Normal patients wait {fifo['normal: average wait (min)']:.0f} → {ai['normal: average wait (min)']:.0f} min.\n\n"
            f"*Simulated with 200 clinic days of held-out test X-rays.*")
    fig = queue_plot(waits, float(minutes_per_read), int(n_scans))
    return text, fig

ABOUT = f'''
**Who it's for:** staff at a clinic without an on-site radiologist, deciding which X-rays get read first.

**Model:** {best_name}, trained on PneumoniaMNIST (chest X-rays of children aged 1–5, one hospital).

**On held-out test X-rays:** caught **{TEST_STATS['recall']:.0%}** of pneumonia cases; wrongly flagged
**{TEST_STATS['false_alarm_rate']:.0%}** of normal scans.

**Not tested on:** adults, other hospitals or other X-ray machines. Expect it to be less accurate there.

**Heatmap:** Grad-CAM shows roughly where the model looked (coarse; not proof of what it "saw").

**Vital-sign rules:** based on WHO guidance for children under 5; they can only raise urgency. They have
**not** been tested on real patient data.
'''

# Check the app logic once without launching the web page
_card, _heat, _text = run_app(Image.open(example_paths[2]), 30, 45, 88, 38.5, False)
print("App check:", "EMERGENCY" in _card, "| has Why section:", "Why" in _card, "| heatmap:", _heat.size if _heat else None, "|", _text[:120])
print("Refusal check:", "CANNOT ASSESS" in run_app(Image.fromarray(np.zeros((IMG_SIZE, IMG_SIZE), np.uint8)), 0, 0, 0, 0, False)[0])
print("Queue check:", run_queue(40, 6, 30)[0][:90]); plt.close("all")
_c, _h, _hist, _facts = check_xray(Image.open(example_paths[2]), 30, 45, 88, 38.5, False)
print("Chat starts empty:", _hist == [], "| text for the voice is in the box:", 'class="speak"' in _c)
_hist2, _spoken2, _ = ask("What should I do next?", _hist, _facts)
print("Chat reply:", _hist2[-1]["content"][:100], "| turns:", len(_hist2))

In [ ]:
HEADER = '''
<div style="text-align:center;padding:2px 0 0">
  <svg width="48" height="48" viewBox="0 0 64 64" role="img" aria-label="lungs symbol">
    <circle cx="32" cy="32" r="31" fill="#0f766e"/>
    <rect x="30" y="9" width="4" height="17" rx="2" fill="#ffffff"/>
    <path d="M32 25 L25 31 M32 25 L39 31" stroke="#ffffff" stroke-width="3" stroke-linecap="round" fill="none"/>
    <path d="M27 29 C18 27 12 39 13 49 C14 54 23 55 28 50 Z" fill="#ffffff" opacity="0.95"/>
    <path d="M37 29 C46 27 52 39 51 49 C50 54 41 55 36 50 Z" fill="#ffffff" opacity="0.95"/>
  </svg>
  <h1 style="margin:2px 0 0;font-size:22px">Pneumonia Triage Assistant</h1>
  <p style="margin:0;color:#64748b;font-size:13px">Sorts chest X-rays so likely pneumonia cases are read first.
  <b>Prototype only, not medical advice. A clinician must read every X-ray.</b></p>
</div>'''

# JavaScript that runs in the viewer's browser (Chrome or Edge recommended)
READ_CARD_JS = '''() => {
  const el = document.querySelector('#condition-card .speak');
  if (!el || !window.speechSynthesis) return;
  speechSynthesis.cancel();
  const u = new SpeechSynthesisUtterance(el.textContent.trim()); u.lang = 'en-US'; u.rate = 1.0;
  speechSynthesis.speak(u);
}'''
READ_ANSWER_JS = READ_CARD_JS.replace('#condition-card', '#last-answer')
STOP_JS = '''() => { if (window.speechSynthesis) speechSynthesis.cancel(); }'''
LISTEN_JS = '''() => new Promise((resolve) => {
  const SR = window.SpeechRecognition || window.webkitSpeechRecognition;
  if (!SR) { alert('Voice input works in Chrome or Edge. Please type your question instead.'); resolve(''); return; }
  if (window.speechSynthesis) speechSynthesis.cancel();
  const r = new SR(); r.lang = 'en-US'; r.interimResults = false; r.maxAlternatives = 1;
  r.onresult = (e) => resolve(e.results[0][0].transcript);
  r.onerror = () => resolve(''); r.onend = () => resolve('');
  r.start();
})'''

if not QUICK_RUN:
    import gradio as gr, inspect
    theme = gr.themes.Soft(primary_hue="teal")
    # Newer Gradio versions take the theme in launch() instead of Blocks()
    APP_CSS = '''
#condition-card, #condition-card .prose, #last-answer, #last-answer .prose {
    border: 0 !important; box-shadow: none !important; background: transparent !important;
    padding: 0 !important; min-height: 0 !important; }
#last-answer { display: none !important; }
#condition-card .speak, #condition-card hr { display: none !important; }
'''
    blocks_kw, launch_kw = {"title": "Pneumonia triage assistant"}, {"share": True}
    (blocks_kw if "theme" in inspect.signature(gr.Blocks.__init__).parameters else launch_kw)["theme"] = theme
    (blocks_kw if "css" in inspect.signature(gr.Blocks.__init__).parameters else launch_kw)["css"] = APP_CSS

    def make_chatbot():
        kw = dict(label="Ask about this result", height=330)
        params = inspect.signature(gr.Chatbot.__init__).parameters
        if "type" in params: kw["type"] = "messages"
        if "placeholder" in params: kw["placeholder"] = "Check an X-ray first, then ask, for example: \"Why is it urgent?\" or \"What should I watch for?\""
        return gr.Chatbot(**kw)

    with gr.Blocks(**blocks_kw) as demo:
        gr.HTML(HEADER)
        facts_state = gr.State("")
        with gr.Tab("Check an X-ray"):
            with gr.Row(equal_height=False):
                # left: the X-ray and the optional patient observations
                with gr.Column(scale=2, min_width=240):
                    image_in = gr.Image(type="pil", label="Chest X-ray", height=230)
                    go = gr.Button("Check X-ray", variant="primary")
                    with gr.Accordion("Patient observations (optional)", open=False):
                        age_in = gr.Number(label="Age (months)", value=None, precision=0)
                        rr_in = gr.Number(label="Breathing rate (breaths per minute)", value=None)
                        spo2_in = gr.Number(label="Oxygen saturation SpO₂ (%)", value=None)
                        temp_in = gr.Number(label="Temperature (°C)", value=None)
                        danger_in = gr.Checkbox(label="General danger sign (can't drink, vomits everything, convulsions, lethargic)")
                    with gr.Accordion("Try a test X-ray", open=False):
                        gr.Examples(examples=example_paths, inputs=image_in, label="")
                # middle: the condition box with the reasons underneath
                with gr.Column(scale=3, min_width=300):
                    card_out = gr.HTML("", elem_id="condition-card")   # appears here after "Check X-ray"
                    with gr.Row():
                        read_btn = gr.Button("🔊 Read aloud", size="sm")
                        stop_btn = gr.Button("⏹ Stop", size="sm")
                    with gr.Accordion("Where the model looked (Grad-CAM)", open=False):
                        heat_out = gr.Image(show_label=False, height=240, interactive=False)
                # right: the chat box, always visible next to the result
                with gr.Column(scale=3, min_width=300):
                    chat = make_chatbot()
                    with gr.Row():
                        question = gr.Textbox(placeholder="Ask, or press 🎤 and speak…", show_label=False, scale=5, container=False)
                        mic = gr.Button("🎤", scale=1, min_width=50)
                        send = gr.Button("Send", variant="primary", scale=1, min_width=70)
                    last_answer = gr.HTML("", elem_id="last-answer")   # hidden text the browser reads aloud
            with gr.Accordion("How reliable is this?", open=False):
                gr.Markdown(ABOUT)
            inputs = [image_in, age_in, rr_in, spo2_in, temp_in, danger_in]
            go.click(check_xray, inputs=inputs, outputs=[card_out, heat_out, chat, facts_state]
                     ).then(None, None, None, js=READ_CARD_JS)
            read_btn.click(None, None, None, js=READ_CARD_JS)
            stop_btn.click(None, None, None, js=STOP_JS)
            ask_io = dict(inputs=[question, chat, facts_state], outputs=[chat, last_answer, question])
            send.click(ask, **ask_io).then(None, None, None, js=READ_ANSWER_JS)
            question.submit(ask, **ask_io).then(None, None, None, js=READ_ANSWER_JS)
            # 🎤: the browser listens, writes what you said into the box, sends it, and reads the answer aloud
            mic.click(None, None, question, js=LISTEN_JS).then(ask, **ask_io).then(None, None, None, js=READ_ANSWER_JS)
        with gr.Tab("Clinic and safety"):
            try:
                _c = robustness.loc["clean (no change)", "recall (pneumonia caught)"]
                _w = robustness["recall (pneumonia caught)"].idxmin()
                _wv = robustness["recall (pneumonia caught)"].min()
                shift_txt = (f"On the clean test set the model catches **{_c:.0%}** of pneumonia cases. Under our "
                             f"simulated 'different source' changes (noise, blur, contrast, brightness, rotation, "
                             f"low resolution, JPEG) the worst case was **{_w}**, where it caught **{_wv:.0%}**.")
            except Exception:
                shift_txt = "Run the robustness section of the notebook to see the numbers."
            gr.Markdown(f'''
### 1. A real setting: a rural clinic with no radiologist
Our user is a **nurse or general doctor** in a small clinic. X-rays are taken on site, but a radiologist reads
them remotely, hours or days later. The tool is built around that:
- **One photo in, one answer out.** Upload the X-ray, optionally type age, breathing rate, oxygen and temperature.
- **It only changes the order of reading.** A clinician still reads every scan.
- **Three labels staff can act on:** 🔴 URGENT (send first, or treat as an emergency), 🟡 UNSURE (a person looks
  soon), 🟢 LIKELY NORMAL (normal queue).
- **Works with little equipment:** the small model runs on an ordinary laptop CPU (see the speed section of the notebook).
- **Plain-language explanation and voice,** because staff are busy and may not read technical output.
- **Queue simulator tab** shows how much sooner pneumonia patients are read when the AI orders the queue.

### 2. Does it still work on X-rays from a different source?
{shift_txt}

**Honest limit:** we only had one public pneumonia dataset (children, one hospital in Guangzhou). We could not
test a real second hospital or adult X-rays, so these changes are a **stand-in**, not proof. The input check also
refuses images that don't look like a chest X-ray. Before real use, the model must be tested on local X-rays.

### 3. How scans it is unsure about are handled, and why
- Between a low and a high score the tool says **UNSURE** instead of forcing a yes or no.
- **Why:** a wrong 'normal' on a sick child is the worst mistake, and a forced guess hides how uncertain the model is.
  UNSURE scans go to a person soon, so the model never has the last word on the hard ones.
- The lower edge is set on validation data so that at least 99% of validation pneumonia cases score above it.
  The cost: more scans for staff to review. We think that is the right trade in a clinic where missing pneumonia is dangerous.
- Blank, noisy or non-X-ray images are **refused**, not guessed.

*This project is a prototype and is not medical advice.*
''')
        with gr.Tab("Queue simulator"):
            gr.Markdown("How much sooner do pneumonia patients get their X-ray read when the AI orders the queue?")
            with gr.Row():
                n_in = gr.Slider(10, 100, value=40, step=5, label="X-rays per day")
                m_in = gr.Slider(2, 20, value=6, step=1, label="Minutes to read one X-ray")
                prev_in = gr.Slider(5, 80, value=30, step=5, label="Share with pneumonia (%)")
            sim_btn = gr.Button("Run simulation", variant="primary")
            sim_text = gr.Markdown()
            sim_plot = gr.Plot()
            sim_btn.click(run_queue, inputs=[n_in, m_in, prev_in], outputs=[sim_text, sim_plot])
    demo.launch(**launch_kw)

## Results to copy into the write-up

Run the cell below and paste its output into the **Results** section of `README.md`. Report the numbers
you actually got, including weak ones.

In [ ]:
try:
    print(summary.to_markdown())      # needs the 'tabulate' package (preinstalled in Colab)
except ImportError:
    print(summary.to_string())
print(f"\nChosen model: {best_name} | threshold {THRESHOLD:.3f} | unsure band {UNSURE_LOW:.4f}–{THRESHOLD:.4f}")
print(f"Dangerous misses (pneumonia → LIKELY NORMAL): {missed_in_normal_pile}/{(y_te==1).sum()}")
print(f"Scans flagged for review first (URGENT + UNSURE): {(t_test != 'LIKELY NORMAL').mean():.0%}")
print("\nRobustness (recall / false alarms per condition):")
print(robustness[["recall (pneumonia caught)", "missed cases (FN)", "false alarms (FP)", "refused by input check"]].to_string())
print(f"\nGrad-CAM heat in image border: {shares.mean():.0%} (even spread = {area_share:.0%})")
print(f"Real X-rays wrongly refused by input check: {clean_refused:.1%}")
print("\nQueue simulation (40 X-rays/day, 6 min each, 30% pneumonia):")
print(queue_summary.to_string())
print("\nLatency:")
print(latency.to_string())
print(f"\n⏱ Total notebook run time: {(time.time() - NOTEBOOK_START) / 60:.1f} minutes")